In [2]:
!python -m pip install --user ipywidgets imageio-ffmpeg

  Using cached imageio_ffmpeg-0.6.0-py3-none-win_amd64.whl.metadata (1.5 kB)
   ---------------------------------------- 0.0/31.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/31.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/31.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/31.2 MB ? eta -:--:--
    --------------------------------------- 0.5/31.2 MB 1.6 MB/s eta 0:00:20
   - -------------------------------------- 1.0/31.2 MB 2.0 MB/s eta 0:00:16
   - -------------------------------------- 1.3/31.2 MB 1.9 MB/s eta 0:00:16
   -- ------------------------------------- 1.6/31.2 MB 1.6 MB/s eta 0:00:19
   --- ------------------------------------ 2.4/31.2 MB 2.2 MB/s eta 0:00:14
   ---- ----------------------------------- 3.1/31.2 MB 2.3 MB/s eta 0:00:13
   ---- ----------------------------------- 3.7/31.2 MB 2.5 MB/s eta 0:00:11
   ---- ----------------------------------- 3.7/31.2 MB 2.5 MB/s eta 0:00:11
   ------ -----------------

In [4]:
import sys

!{sys.executable} -m pip install --user imageio-ffmpeg

In [1]:
import cv2
import os
import tempfile
import subprocess
import imageio_ffmpeg
from datetime import datetime
from ultralytics import YOLO

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Video
from base64 import b64encode

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
MODEL_NAME = "yolov8n.pt"

model = YOLO(MODEL_NAME)

print("YOLOv8 model loaded successfully.")

YOLOv8 model loaded successfully.


In [3]:
def convert_to_playable_mp4(input_path, output_path):
    ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()

    subprocess.run([
        ffmpeg_path,
        "-y",
        "-i", input_path,
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        output_path
    ], check=True)


def display_video(video_path, width=500):
    with open(video_path, "rb") as video_file:
        video_data = video_file.read()

    encoded_video = b64encode(video_data).decode("utf-8")

    return HTML(f"""
    <video width="{width}" controls>
        <source src="data:video/mp4;base64,{encoded_video}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
    """)


def process_video(input_video_path, confidence_threshold, progress_bar, status_label):
    if not os.path.exists(input_video_path):
        raise FileNotFoundError(f"Video file not found: {input_video_path}")

    cap = cv2.VideoCapture(input_video_path)

    if not cap.isOpened():
        raise RuntimeError("Error: Could not open video file.")

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps == 0:
        fps = 25

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    raw_output_path = f"border_monitoring_raw_{timestamp}.mp4"
    playable_output_path = f"border_monitoring_output_{timestamp}.mp4"

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    out = cv2.VideoWriter(
        raw_output_path,
        fourcc,
        fps,
        (frame_width, frame_height)
    )

    if not out.isOpened():
        cap.release()
        raise RuntimeError("Error: Could not create output video file.")

    frame_number = 0
    total_human_detections = 0
    alert_detected = False
    max_humans_in_frame = 0

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        frame_number += 1
        human_count = 0

        results = model.track(
            frame,
            persist=True,
            classes=[0],
            conf=confidence_threshold,
            tracker="bytetrack.yaml",
            verbose=False
        )

        if results and results[0].boxes is not None:
            for box in results[0].boxes:
                human_count += 1
                total_human_detections += 1
                alert_detected = True

                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                confidence = float(box.conf[0])

                track_id = None
                if box.id is not None:
                    track_id = int(box.id[0])

                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

                label = f"Human {confidence:.2f}"

                if track_id is not None:
                    label = f"ID {track_id} | Human {confidence:.2f}"

                cv2.putText(
                    frame,
                    label,
                    (x1, max(y1 - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2
                )

        max_humans_in_frame = max(max_humans_in_frame, human_count)

        cv2.putText(
            frame,
            f"Human Count: {human_count}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255, 255, 0),
            2
        )

        if human_count > 0:
            cv2.putText(
                frame,
                "ALERT: Human Detected!",
                (20, 85),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                3
            )

        out.write(frame)

        if total_frames > 0:
            progress = int((frame_number / total_frames) * 100)
            progress_bar.value = min(progress, 100)

        status_label.value = f"Processing frame {frame_number} of {total_frames}"

    cap.release()
    out.release()

    status_label.value = "Converting video to playable format..."
    convert_to_playable_mp4(raw_output_path, playable_output_path)

    return {
        "output_video": playable_output_path,
        "raw_output_video": raw_output_path,
        "total_frames": total_frames,
        "fps": fps,
        "width": frame_width,
        "height": frame_height,
        "total_human_detections": total_human_detections,
        "max_humans_in_frame": max_humans_in_frame,
        "alert_detected": alert_detected
    }

print("Processing function created successfully.")

Processing function created successfully.


In [4]:
title = widgets.HTML("""
<h2 style="color:#1f4e79;">Border Area Monitoring System</h2>
<h4>AI-Powered Drone Human Detection and Motion Tracking</h4>
""")

instruction = widgets.HTML("""
<p>Select your drone surveillance video, set the confidence threshold, then click <b>Start Detection</b>.</p>
""")

video_path_input = widgets.Text(
    value=r"C:\Users\RAYMONDMUSTWIN\Documents\Surveillance project\demo_video.mp4",
    placeholder="Enter full video path here",
    description="Video Path:",
    layout=widgets.Layout(width="90%")
)

confidence_slider = widgets.FloatSlider(
    value=0.40,
    min=0.10,
    max=1.00,
    step=0.05,
    description="Confidence:",
    readout_format=".2f",
    layout=widgets.Layout(width="60%")
)

start_button = widgets.Button(
    description="Start Detection",
    button_style="success",
    layout=widgets.Layout(width="200px")
)

progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Progress:",
    bar_style="info",
    layout=widgets.Layout(width="70%")
)

status_label = widgets.Label(value="Waiting to start...")

output_area = widgets.Output()

ui = widgets.VBox([
    title,
    instruction,
    video_path_input,
    confidence_slider,
    start_button,
    progress_bar,
    status_label,
    output_area
])

display(ui)

In [5]:
def start_detection(button):
    progress_bar.value = 0
    status_label.value = "Starting detection..."

    with output_area:
        clear_output()

        input_video_path = video_path_input.value.strip()
        confidence_threshold = confidence_slider.value

        try:
            print("Input Video:", input_video_path)
            print("Confidence Threshold:", confidence_threshold)
            print("Processing started. Please wait...")

            result = process_video(
                input_video_path,
                confidence_threshold,
                progress_bar,
                status_label
            )

            clear_output()

            print("Video processing completed successfully.")
            print()
            print("RESULT SUMMARY")
            print("Total Frames:", result["total_frames"])
            print("Video Width:", result["width"])
            print("Video Height:", result["height"])
            print("FPS:", round(result["fps"], 2))
            print("Total Human Detections:", result["total_human_detections"])
            print("Maximum Humans in One Frame:", result["max_humans_in_frame"])
            print("Alert Status:", "Human Detected" if result["alert_detected"] else "No Human Detected")
            print("Output Video:", os.path.abspath(result["output_video"]))

            display(display_video(result["output_video"], width=500))

            status_label.value = "Completed successfully."
            progress_bar.value = 100

        except Exception as error:
            clear_output()
            status_label.value = "An error occurred."
            print("Error:", error)


start_button.on_click(start_detection)

print("User interface is ready. Click Start Detection to begin.")

User interface is ready. Click Start Detection to begin.
